# 📘 Guide 09: AI APIs, costs and Ollama

**Who it's for:** you'd like to know what a lab will cost before you run it, or you'd rather run models for free on your own machine. **Time:** about 25 minutes.

Good news first: the whole course costs a few dollars with small models. This guide shows you where the money goes and how to keep it low, or at zero.

### 🎯 You'll learn
1. Tokens: the unit everything is priced in
2. Estimating the cost of grading a whole class
3. `BUDGET_MODE` and choosing models in `.env`
4. Running free local models with Ollama
5. Swapping providers without changing code

## Part 1: Tokens

Models don't read words; they read **tokens**, chunks of roughly 4 characters of English (about ¾ of a word). You pay for:
- **input tokens**: your prompt, the rubric, retrieved course notes, the student's answer
- **output tokens**: what the model writes back (usually priced 4–8× higher)

A quick estimate is good enough for budgeting:

In [ ]:
def estimate_tokens(text: str) -> int:
    """Rough rule of thumb: ~4 characters per token for English."""
    return max(1, round(len(text) / 4))


rubric = "Award 2 marks for defining a stack as LIFO, 2 for push/pop, 1 for a real-world example. " * 3
answer = "A stack is a data structure where the last element pushed is the first popped, like plates. " * 4
print(f"Rubric ≈ {estimate_tokens(rubric)} tokens, answer ≈ {estimate_tokens(answer)} tokens")

try:
    import tiktoken  # an exact tokenizer for OpenAI models, if installed

    enc = tiktoken.get_encoding("o200k_base")
    print(f"tiktoken says the answer is exactly {len(enc.encode(answer))} tokens")
except Exception:
    print("(tiktoken not available offline: the estimate is fine for budgeting)")

## Part 2: What does grading a class cost?

Prices are quoted **per million tokens** and change often, so check your provider's pricing page. The numbers below are illustrative placeholders: edit them to match today's prices.

In [ ]:
# ✋ TODO: update these from your provider's pricing page (USD per 1M tokens)
PRICES = {
    "small model": {"input": 0.25, "output": 2.00},
    "large model": {"input": 1.25, "output": 10.00},
}

students, questions = 30, 5
input_per_call = 2_500  # rubric + retrieved notes + answer + instructions
output_per_call = 400  # marks, evidence and feedback

calls = students * questions
for name, p in PRICES.items():
    cost = calls * (input_per_call * p["input"] + output_per_call * p["output"]) / 1_000_000
    print(f"{name:>12}: {calls} calls → ${cost:.2f} for Asha's whole CS201 class")

That's why the course uses a **small model** for everyday work and saves the larger one for the hardest step (grading).

## Part 3: `BUDGET_MODE` and models in `.env`

GradeWise never hard-codes a model name. They all live in `.env`, written as `provider:model`:

```
CHAT_MODEL=openai:gpt-5.4-mini      # everyday tasks
SMART_MODEL=openai:gpt-5.5          # grading
VISION_MODEL=openai:gpt-5.5         # reading handwriting
BUDGET_MODE=false
```

Set `BUDGET_MODE=true` and every tier falls back to `CHAT_MODEL`. Results are a little less sharp, but it costs a fraction. Let's see the logic from `gradewise/config.py` (this just reads settings; it makes no API calls):

In [ ]:
from gradewise.config import Settings

for budget in (False, True):
    s = Settings(budget_mode=budget)
    print(f"BUDGET_MODE={budget!s:<5} → grading uses {s.grading_model}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Protect your wallet</h2>
            <span style="color:#22c55e;">Set a <b>monthly spending limit</b> on your provider's billing page before you start. It's the single best protection against a runaway loop. For a course like this, $5–10 is plenty.</span>
        </td>
    </tr>
</table>

## Part 4: Free local models with Ollama

[Ollama](https://ollama.com) runs open models on your own computer: no key, no cost, and your students' data never leaves the machine (a real plus for schools).

Install it:
- **Windows / Mac:** download the installer from ollama.com
- **Linux:** `curl -fsSL https://ollama.com/install.sh | sh`

Then, in a terminal (same on every OS):
```bash
ollama pull llama3.2          # download a small model (~2 GB)
ollama run llama3.2           # chat with it in the terminal; /bye to quit
ollama list                   # what you have installed
```

And point GradeWise at it in `.env`:
```
CHAT_MODEL=ollama:llama3.2
BUDGET_MODE=true
```

Local models are slower and less capable than frontier ones, especially at reading handwriting. They're great for following along with weeks 1–3 for free.

In [ ]:
import httpx

# 💡 Is Ollama running? This only talks to your own machine, and it's fine if it isn't installed.
try:
    r = httpx.get("http://localhost:11434/api/tags", timeout=1.0)
    models = [m["name"] for m in r.json().get("models", [])]
    print("✅ Ollama is running. Models:", models or "none pulled yet")
except Exception:
    print("ℹ️ Ollama isn't running on this machine (that's fine). Install it from ollama.com to use free local models.")

## Part 5: Same code, any provider

Because labs use LangChain's `init_chat_model`, switching provider is a `.env` change. In code it looks like this (shown, not run, since it needs a key or Ollama):

```python
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-5.4-mini")             # or "anthropic:...", "google_genai:...", "ollama:llama3.2"
reply = llm.invoke("In one sentence, what is a stack?")
print(reply.content)
print(reply.usage_metadata)   # 💡 input_tokens, output_tokens: the numbers you pay for
```

See `setup/SWAP_PROVIDER.md` for every supported provider.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Try it</h2>
            <span style="color:#f59e0b;">Update <code>PRICES</code> in Part 2 with today's real prices for the models in your <code>.env</code>, and work out the cost of grading 120 students × 8 questions. Then, if you can, install Ollama, pull a small model and re-run the check cell.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• OpenAI pricing: <a href="https://openai.com/api/pricing/">openai.com/api/pricing/</a><br/>• OpenAI tokenizer playground: <a href="https://platform.openai.com/tokenizer">platform.openai.com/tokenizer</a><br/>• Ollama: <a href="https://ollama.com">ollama.com</a><br/>• LangChain init_chat_model: <a href="https://docs.langchain.com/oss/python/langchain/models">docs.langchain.com/oss/python/langchain/models</a></span>
        </td>
    </tr>
</table>